E-6. Estragga tutti i 20 pcap, unisca le corrispondenti sequenze in un unico dataframe con una colonna label_20 e una label_2. Riporti la
tabella dei conteggi per classe. Ci aspettiamo forte sbilanciamento: per il pre-training, faremo un sottocampionamento a classi bilanciate, la
scelta più semplice e la più difendibile — scriviamolo nel config.yaml.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
BASE_DIR = "/content/drive/MyDrive/Neural Collapse/USTC-TFC2016"

RAW_DIR = BASE_DIR + "/raw"
PROCESSED_DIR = BASE_DIR + "/processed"

print("Cartella principale:", BASE_DIR)
print("Cartella PCAP:", RAW_DIR)
print("Cartella risultati:", PROCESSED_DIR)

Cartella principale: /content/drive/MyDrive/Neural Collapse/USTC-TFC2016
Cartella PCAP: /content/drive/MyDrive/Neural Collapse/USTC-TFC2016/raw
Cartella risultati: /content/drive/MyDrive/Neural Collapse/USTC-TFC2016/processed


In [ ]:
!git clone https://github.com/davidyslu/USTC-TFC2016.git /content/ustc-tfc2016

Cloning into '/content/ustc-tfc2016'...
remote: Enumerating objects: 37, done.
remote: Total 37 (delta 0), reused 0 (delta 0), pack-reused 37 (from 1)
Receiving objects: 100% (37/37), 314.88 MiB | 22.93 MiB/s, done.
Resolving deltas: 100% (5/5), done.
Updating files: 100% (22/22), done.


In [ ]:
!apt-get -qq update
!apt-get -qq install p7zip-full

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
import os
import subprocess

for categoria in ["Benign", "Malware"]:
    cartella = f"/content/ustc-tfc2016/{categoria}"

    for file in os.listdir(cartella):
        if file.endswith(".7z"):
            percorso = os.path.join(cartella, file)

            print(f"Estrazione di {file}...")

            subprocess.run([
                "7z",
                "x",
                percorso,
                f"-o{cartella}",
                "-y"
            ], check=True)

print("Estrazione completata!")

Estrazione di SMB.7z...
Estrazione di Weibo.7z...
Estrazione di Htbot.7z...
Estrazione di Geodo.7z...
Estrazione di Virut.7z...
Estrazione di Nsis-ay.7z...
Estrazione di Cridex.7z...
Estrazione di Shifu.7z...
Estrazione di Neris.7z...
Estrazione completata!


In [ ]:
import shutil
import os

RAW_DIR = "/content/drive/MyDrive/Neural Collapse/USTC-TFC2016/raw"

os.makedirs(RAW_DIR, exist_ok=True)

print("Cartella raw pronta:")
print(RAW_DIR)

Cartella raw pronta:
/content/drive/MyDrive/Neural Collapse/USTC-TFC2016/raw


In [ ]:
import shutil
import os

SOURCE_DIR = "/content/ustc-tfc2016"

for categoria in ["Benign", "Malware"]:

    sorgente = os.path.join(SOURCE_DIR, categoria)
    destinazione = os.path.join(RAW_DIR, categoria)

    os.makedirs(destinazione, exist_ok=True)

    for radice, cartelle, files in os.walk(sorgente):

        for file in files:

            if file.endswith(".pcap"):

                percorso_sorgente = os.path.join(radice, file)

                # Manteniamo la struttura delle sottocartelle
                percorso_relativo = os.path.relpath(
                    percorso_sorgente,
                    sorgente
                )

                percorso_destinazione = os.path.join(
                    destinazione,
                    percorso_relativo
                )

                os.makedirs(
                    os.path.dirname(percorso_destinazione),
                    exist_ok=True
                )

                shutil.copy2(
                    percorso_sorgente,
                    percorso_destinazione
                )

                print("Copiato:", percorso_relativo)

print("\nCopia completata!")

Copiato: WorldOfWarcraft.pcap
Copiato: FTP.pcap
Copiato: Gmail.pcap
Copiato: Facetime.pcap
Copiato: BitTorrent.pcap
Copiato: MySQL.pcap
Copiato: Skype.pcap
Copiato: Outlook.pcap
Copiato: Weibo/Weibo-4.pcap
Copiato: Weibo/Weibo-1.pcap
Copiato: Weibo/Weibo-3.pcap
Copiato: Weibo/Weibo-2.pcap
Copiato: SMB/SMB-1.pcap
Copiato: SMB/SMB-2.pcap
Copiato: Neris.pcap
Copiato: Miuref.pcap
Copiato: Htbot.pcap
Copiato: Tinba.pcap
Copiato: Virut.pcap
Copiato: Shifu.pcap
Copiato: Nsis-ay.pcap
Copiato: Zeus.pcap
Copiato: Geodo.pcap
Copiato: Cridex.pcap

Copia completata!


In [ ]:
!pip install -q nfstream

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.0/195.0 kB 12.2 MB/s eta 0:00:00


In [ ]:
from nfstream import NFStreamer

print("NFStream installato correttamente")

NFStream installato correttamente


In [ ]:
pcap_info = []

for categoria in ["Benign", "Malware"]:

    cartella_categoria = os.path.join(RAW_DIR, categoria)

    for radice, cartelle, files in os.walk(cartella_categoria):

        for file in files:

            if file.endswith(".pcap"):

                percorso = os.path.join(radice, file)

                # Determiniamo la classe dal percorso
                percorso_relativo = os.path.relpath(
                    percorso,
                    cartella_categoria
                )

                # La prima parte del percorso identifica la classe
                parti = percorso_relativo.split(os.sep)

                if len(parti) > 1:
                    label_20 = parti[0]
                else:
                    label_20 = os.path.splitext(file)[0]

                label_2 = categoria

                pcap_info.append({
                    "path": percorso,
                    "label_20": label_20,
                    "label_2": label_2
                })

print("Numero totale di PCAP:", len(pcap_info))

Numero totale di PCAP: 24


In [ ]:
TEMP_PROCESSED_DIR = "/content/ustc-tfc2016/processed"

os.makedirs(TEMP_PROCESSED_DIR, exist_ok=True)

print(TEMP_PROCESSED_DIR)

/content/ustc-tfc2016/processed


In [ ]:
!pip install -q pyarrow

In [ ]:
import os
import gc
import ast
import pandas as pd
from nfstream import NFStreamer

TEMP_PROCESSED_DIR = "/content/ustc-tfc2016/processed"

os.makedirs(TEMP_PROCESSED_DIR, exist_ok=True)

# Colonne che NON useremo nel modello
colonne_da_eliminare = [
    "id",
    "expiration_id",
    "src_ip",
    "src_mac",
    "src_oui",
    "src_port",
    "dst_ip",
    "dst_mac",
    "dst_oui",
    "dst_port",
    "vlan_id",
    "tunnel_id",
    "bidirectional_first_seen_ms",
    "bidirectional_last_seen_ms",
    "src2dst_first_seen_ms",
    "src2dst_last_seen_ms",
    "dst2src_first_seen_ms",
    "dst2src_last_seen_ms"
]

totale = len(pcap_info)

print(f"PCAP da elaborare: {totale}\n")

for indice, info in enumerate(pcap_info, start=1):

    print("=" * 70)
    print(f"[{indice}/{totale}] {info['label_20']}")
    print(f"File: {os.path.basename(info['path'])}")
    print("=" * 70)

    # 1. Estrazione NFStream
    streamer = NFStreamer(
        source=info["path"],
        statistical_analysis=True,
        splt_analysis=20,
        n_dissections=0
    )

    df = streamer.to_pandas()

    print(f"Flussi estratti: {len(df)}")

    # 2. Conversione SPLT
    df["splt_direction"] = df["splt_direction"].apply(ast.literal_eval)
    df["splt_ps"] = df["splt_ps"].apply(ast.literal_eval)
    df["splt_piat_ms"] = df["splt_piat_ms"].apply(ast.literal_eval)

    # 3. Aggiunta delle label
    df["label_20"] = info["label_20"]
    df["label_2"] = info["label_2"]

    # 4. Rimuoviamo le informazioni
    #    che non useremo nel modello
    colonne_presenti = [
        col for col in colonne_da_eliminare
        if col in df.columns
    ]

    df = df.drop(columns=colonne_presenti)

    # 5. Salvataggio temporaneo
    nome_output = f"{indice:02d}_{info['label_20']}.parquet"

    percorso_output = os.path.join(
        TEMP_PROCESSED_DIR,
        nome_output
    )

    df.to_parquet(
        percorso_output,
        index=False
    )

    print(f"Salvato: {nome_output}")
    print(f"Colonne finali: {len(df.columns)}")

    # 6. Liberiamo la memoria

    del streamer
    del df

    gc.collect()

    print("Memoria liberata.\n")

print("=" * 70)
print("ESTRAZIONE COMPLETATA")
print("=" * 70)

PCAP da elaborare: 24

[1/24] WorldOfWarcraft
File: WorldOfWarcraft.pcap
Flussi estratti: 7883
Salvato: 01_WorldOfWarcraft.parquet
Colonne finali: 64
Memoria liberata.

[2/24] FTP
File: FTP.pcap
Flussi estratti: 101037
Salvato: 02_FTP.parquet
Colonne finali: 64
Memoria liberata.

[3/24] Gmail
File: Gmail.pcap
Flussi estratti: 8629
Salvato: 03_Gmail.parquet
Colonne finali: 64
Memoria liberata.

[4/24] Facetime
File: Facetime.pcap
Flussi estratti: 6000
Salvato: 04_Facetime.parquet
Colonne finali: 64
Memoria liberata.

[5/24] BitTorrent
File: BitTorrent.pcap
Flussi estratti: 7517
Salvato: 05_BitTorrent.parquet
Colonne finali: 64
Memoria liberata.

[6/24] MySQL
File: MySQL.pcap
Flussi estratti: 86089
Salvato: 06_MySQL.parquet
Colonne finali: 64
Memoria liberata.

[7/24] Skype
File: Skype.pcap
Flussi estratti: 6321
Salvato: 07_Skype.parquet
Colonne finali: 64
Memoria liberata.

[8/24] Outlook
File: Outlook.pcap
Flussi estratti: 7524
Salvato: 08_Outlook.parquet
Colonne finali: 64
Memoria lib

In [ ]:
conteggi = []

for file in files_parquet:

    df_temp = pd.read_parquet(
        file,
        columns=["label_20", "label_2"]
    )

    conteggi.append({
        "file": os.path.basename(file),
        "flussi": len(df_temp),
        "label_20": df_temp["label_20"].iloc[0],
        "label_2": df_temp["label_2"].iloc[0]
    })

    del df_temp

conteggi_df = pd.DataFrame(conteggi)

print(conteggi_df.to_string(index=False))

print("\nTotale flussi:")
print(conteggi_df["flussi"].sum())

                      file  flussi        label_20 label_2
01_WorldOfWarcraft.parquet    7883 WorldOfWarcraft  Benign
            02_FTP.parquet  101037             FTP  Benign
          03_Gmail.parquet    8629           Gmail  Benign
       04_Facetime.parquet    6000        Facetime  Benign
     05_BitTorrent.parquet    7517      BitTorrent  Benign
          06_MySQL.parquet   86089           MySQL  Benign
          07_Skype.parquet    6321           Skype  Benign
        08_Outlook.parquet    7524         Outlook  Benign
          09_Weibo.parquet    4935           Weibo  Benign
          10_Weibo.parquet   24953           Weibo  Benign
          11_Weibo.parquet    5028           Weibo  Benign
          12_Weibo.parquet    5034           Weibo  Benign
            13_SMB.parquet   32661             SMB  Benign
            14_SMB.parquet    6276             SMB  Benign
          15_Neris.parquet   39429           Neris Malware
         16_Miuref.parquet   14706          Miuref Malwa

In [ ]:
conteggi_classi = (
    conteggi_df
    .groupby(["label_2", "label_20"])["flussi"]
    .sum()
    .reset_index()
    .sort_values(["label_2", "flussi"], ascending=[True, False])
)

print(conteggi_classi.to_string(index=False))

label_2        label_20  flussi
 Benign             FTP  101037
 Benign           MySQL   86089
 Benign           Weibo   39950
 Benign             SMB   38937
 Benign           Gmail    8629
 Benign WorldOfWarcraft    7883
 Benign         Outlook    7524
 Benign      BitTorrent    7517
 Benign           Skype    6321
 Benign        Facetime    6000
Malware          Cridex   61699
Malware           Geodo   50464
Malware           Neris   39429
Malware           Virut   37413
Malware          Miuref   14706
Malware            Zeus   11755
Malware           Shifu   11260
Malware           Tinba   10448
Malware           Htbot    8107
Malware         Nsis-ay    7381


In [ ]:
tabella_conteggi = conteggi_classi.pivot(
    index="label_20",
    columns="label_2",
    values="flussi"
).fillna(0).astype(int)

print(tabella_conteggi)

label_2          Benign  Malware
label_20                        
BitTorrent         7517        0
Cridex                0    61699
FTP              101037        0
Facetime           6000        0
Geodo                 0    50464
Gmail              8629        0
Htbot                 0     8107
Miuref                0    14706
MySQL             86089        0
Neris                 0    39429
Nsis-ay               0     7381
Outlook            7524        0
SMB               38937        0
Shifu                 0    11260
Skype              6321        0
Tinba                 0    10448
Virut                 0    37413
Weibo             39950        0
WorldOfWarcraft    7883        0
Zeus                  0    11755


In [ ]:
print("Inizio unione dei 24 file...")

dataframes = []

for file in files_parquet:
    print("Caricamento:", os.path.basename(file))

    df_temp = pd.read_parquet(file)
    dataframes.append(df_temp)

df_all = pd.concat(
    dataframes,
    ignore_index=True
)

del dataframes

print("\nUnione completata!")
print("Numero di flussi:", len(df_all))
print("Numero di colonne:", len(df_all.columns))

Inizio unione dei 24 file...
Caricamento: 01_WorldOfWarcraft.parquet
Caricamento: 02_FTP.parquet
Caricamento: 03_Gmail.parquet
Caricamento: 04_Facetime.parquet
Caricamento: 05_BitTorrent.parquet
Caricamento: 06_MySQL.parquet
Caricamento: 07_Skype.parquet
Caricamento: 08_Outlook.parquet
Caricamento: 09_Weibo.parquet
Caricamento: 10_Weibo.parquet
Caricamento: 11_Weibo.parquet
Caricamento: 12_Weibo.parquet
Caricamento: 13_SMB.parquet
Caricamento: 14_SMB.parquet
Caricamento: 15_Neris.parquet
Caricamento: 16_Miuref.parquet
Caricamento: 17_Htbot.parquet
Caricamento: 18_Tinba.parquet
Caricamento: 19_Virut.parquet
Caricamento: 20_Shifu.parquet
Caricamento: 21_Nsis-ay.parquet
Caricamento: 22_Zeus.parquet
Caricamento: 23_Geodo.parquet
Caricamento: 24_Cridex.parquet

Unione completata!
Numero di flussi: 562549
Numero di colonne: 64


In [ ]:
PROCESSED_DRIVE = "/content/drive/MyDrive/Neural Collapse/USTC-TFC2016/processed"

os.makedirs(PROCESSED_DRIVE, exist_ok=True)

DATASET_PATH = os.path.join(
    PROCESSED_DRIVE,
    "ustc_tfc2016_all.parquet"
)

print("Salvataggio in corso...")
print(DATASET_PATH)

df_all.to_parquet(
    DATASET_PATH,
    index=False
)

print("\nDataset salvato correttamente!")

Salvataggio in corso...
/content/drive/MyDrive/Neural Collapse/USTC-TFC2016/processed/ustc_tfc2016_all.parquet


NameError: name 'df_all' is not defined

In [ ]:
df_check = pd.read_parquet(DATASET_PATH)

print("File letto correttamente!")
print("Flussi:", len(df_check))
print("Colonne:", len(df_check.columns))

print("\nClassi:", df_check["label_20"].nunique())
print("Label binarie:", df_check["label_2"].nunique())

File letto correttamente!
Flussi: 562549
Colonne: 64

Classi: 20
Label binarie: 2


In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np

# Seed fissato e riproducibile
SEED = 42

# Indici di tutte le righe
indices = np.arange(len(df_check))

# 1. Separiamo TEST (10%) + resto (90%)

idx_rest, idx_test = train_test_split(
    indices,
    test_size=0.10,
    random_state=SEED,
    stratify=df_check["label_20"]
)

# 2. Dal restante 90% ricaviamo PROBE (5% del totale)

# 5% del totale / 90% rimasto = 0.055555...
probe_fraction = 0.05 / 0.90

idx_rest, idx_probe = train_test_split(
    idx_rest,
    test_size=probe_fraction,
    random_state=SEED,
    stratify=df_check.iloc[idx_rest]["label_20"]
)

# 3. Dal restante 85% separiamo VALIDATION (15% totale)

val_fraction = 0.15 / 0.85

idx_train, idx_val = train_test_split(
    idx_rest,
    test_size=val_fraction,
    random_state=SEED,
    stratify=df_check.iloc[idx_rest]["label_20"]
)

# Controllo

print("TRAIN:", len(idx_train))
print("VAL:  ", len(idx_val))
print("TEST: ", len(idx_test))
print("PROBE:", len(idx_probe))

print("\nTotale:",
      len(idx_train) + len(idx_val) + len(idx_test) + len(idx_probe))

TRAIN: 393783
VAL:   84383
TEST:  56255
PROBE: 28128

Totale: 562549


In [ ]:
df_check["split"] = ""

df_check.loc[idx_train, "split"] = "train"
df_check.loc[idx_val, "split"] = "val"
df_check.loc[idx_test, "split"] = "test"
df_check.loc[idx_probe, "split"] = "probe"

print(df_check["split"].value_counts())

split
train    393783
val       84383
test      56255
probe     28128
Name: count, dtype: int64


In [ ]:
FINAL_DATASET_PATH = (
    "/content/drive/MyDrive/Neural Collapse/"
    "USTC-TFC2016/processed/ustc_tfc2016_all_with_split.parquet"
)

print("Salvataggio dataset con split...")

df_check.to_parquet(
    FINAL_DATASET_PATH,
    index=False
)

print("\nSalvataggio completato!")
print(FINAL_DATASET_PATH)

Salvataggio dataset con split...

Salvataggio completato!
/content/drive/MyDrive/Neural Collapse/USTC-TFC2016/processed/ustc_tfc2016_all_with_split.parquet


In [ ]:
import numpy as np

# Numero di elementi realmente presenti nella sequenza
# -1 indica padding
numero_pacchetti = df_check["splt_ps"].apply(
    lambda x: np.sum(np.array(x) != -1)
)

print("Statistiche numero pacchetti reali:")
print(numero_pacchetti.describe())

print("\nDistribuzione dei primi valori:")
print(numero_pacchetti.value_counts().sort_index().head(30))

print("\nFlussi con meno di 20 pacchetti:",
      (numero_pacchetti < 20).sum())

print("Flussi con 20 pacchetti:",
      (numero_pacchetti == 20).sum())

Statistiche numero pacchetti reali:
count    562549.000000
mean          6.572752
std           6.180992
min           1.000000
25%           2.000000
50%           3.000000
75%           9.000000
max          20.000000
Name: splt_ps, dtype: float64

Distribuzione dei primi valori:
splt_ps
1      13822
2     172528
3     133964
4       9280
5      21764
6      23440
7      20166
8       6739
9      30922
10      8136
11      1723
12     31616
13      3800
14      1233
15      1902
16      1003
17      1672
18      9008
19      1360
20     68471
Name: count, dtype: int64

Flussi con meno di 20 pacchetti: 494078
Flussi con 20 pacchetti: 68471


In [ ]:
def crea_mask(x):
    x = np.asarray(x)
    return (x != -1).astype(np.int8)

df_check["splt_mask"] = df_check["splt_ps"].apply(crea_mask)

print("Maschera creata.")

print("\nEsempio:")
print("SPLT PS:")
print(df_check["splt_ps"].iloc[0])

print("\nMASK:")
print(df_check["splt_mask"].iloc[0])

Maschera creata.

Esempio:
SPLT PS:
[ 75  70 238  70  70  70  70  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1
  -1  -1]

MASK:
[1 1 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0]


In [ ]:
controllo_mask = (
    df_check["splt_mask"]
    .apply(len)
    .value_counts()
)

print("Lunghezza maschere:")
print(controllo_mask)

totale_mask = df_check["splt_mask"].apply(np.sum).sum()

print("\nPacchetti reali dalla maschera:", totale_mask)
print("Pacchetti reali calcolati prima:", numero_pacchetti.sum())

print("\nMaschera corretta:",
      totale_mask == numero_pacchetti.sum())

Lunghezza maschere:
splt_mask
20    562549
Name: count, dtype: int64

Pacchetti reali dalla maschera: 3697495
Pacchetti reali calcolati prima: 3697495

Maschera corretta: True


In [ ]:
def log1p_solo_reali(x, mask):
    x = np.asarray(x, dtype=np.float32).copy()
    mask = np.asarray(mask, dtype=bool)

    x[mask] = np.log1p(x[mask])

    return x


df_check["splt_ps_log"] = df_check.apply(
    lambda row: log1p_solo_reali(
        row["splt_ps"],
        row["splt_mask"]
    ),
    axis=1
)

df_check["splt_piat_ms_log"] = df_check.apply(
    lambda row: log1p_solo_reali(
        row["splt_piat_ms"],
        row["splt_mask"]
    ),
    axis=1
)

print("log1p completato.")

log1p completato.


In [ ]:
def converti_direzione(x, mask):
    x = np.asarray(x, dtype=np.int8).copy()
    mask = np.asarray(mask, dtype=bool)

    # Solo sui pacchetti reali
    x[mask & (x == 0)] = -1
    x[mask & (x == 1)] = 1

    # Il padding rimane -1
    x[~mask] = -1

    return x


df_check["splt_direction_encoded"] = df_check.apply(
    lambda row: converti_direzione(
        row["splt_direction"],
        row["splt_mask"]
    ),
    axis=1
)

print("Conversione della direzione completata.")

Conversione della direzione completata.


In [ ]:
# Selezioniamo SOLO il TRAIN
df_train = df_check[df_check["split"] == "train"]

print("Flussi utilizzati per le statistiche:", len(df_train))


def calcola_statistiche_train(colonna, mask_colonna):
    valori_reali = []

    for x, mask in zip(
        df_train[colonna],
        df_train[mask_colonna]
    ):
        x = np.asarray(x, dtype=np.float32)
        mask = np.asarray(mask, dtype=bool)

        valori_reali.append(x[mask])

    valori_reali = np.concatenate(valori_reali)

    media = np.mean(valori_reali)
    std = np.std(valori_reali)

    return media, std


media_ps, std_ps = calcola_statistiche_train(
    "splt_ps_log",
    "splt_mask"
)

media_piat, std_piat = calcola_statistiche_train(
    "splt_piat_ms_log",
    "splt_mask"
)


print("\nSTATISTICHE TRAIN")
print("----------------------------")

print("Dimensioni pacchetti:")
print("Media =", media_ps)
print("Std   =", std_ps)

print("\nInterarrivi:")
print("Media =", media_piat)
print("Std   =", std_piat)

Flussi utilizzati per le statistiche: 393783

STATISTICHE TRAIN
----------------------------
Dimensioni pacchetti:
Media = 5.514297
Std   = 1.4752505

Interarrivi:
Media = 1.2778752
Std   = 2.654461


In [ ]:
def standardizza_sequenza(x, mask, media, std):
    x = np.asarray(x, dtype=np.float32).copy()
    mask = np.asarray(mask, dtype=bool)

    # Standardizziamo SOLO i valori reali
    x[mask] = (x[mask] - media) / std

    # Il padding rimane -1
    x[~mask] = -1

    return x


df_check["splt_ps_std"] = df_check.apply(
    lambda row: standardizza_sequenza(
        row["splt_ps_log"],
        row["splt_mask"],
        media_ps,
        std_ps
    ),
    axis=1
)

df_check["splt_piat_ms_std"] = df_check.apply(
    lambda row: standardizza_sequenza(
        row["splt_piat_ms_log"],
        row["splt_mask"],
        media_piat,
        std_piat
    ),
    axis=1
)

print("Standardizzazione completata.")

Standardizzazione completata.


In [ ]:
# Controllo globale del preprocessing

errori = 0

for i in range(len(df_check)):

    mask = np.asarray(df_check["splt_mask"].iloc[i], dtype=bool)
    ps = np.asarray(df_check["splt_ps_std"].iloc[i])
    piat = np.asarray(df_check["splt_piat_ms_std"].iloc[i])

    # Il padding deve essere sempre -1
    if not np.all(ps[~mask] == -1):
        errori += 1
        break

    if not np.all(piat[~mask] == -1):
        errori += 1
        break

print("Controllo globale completato.")
print("Errori trovati:", errori)

Controllo globale completato.
Errori trovati: 0


In [ ]:
import os

# Copia del dataframe con le colonne definitive
df_final = df_check.copy()

# Sostituiamo le sequenze originali con quelle preprocessate
df_final["splt_direction"] = df_final["splt_direction_encoded"]
df_final["splt_ps"] = df_final["splt_ps_std"]
df_final["splt_piat_ms"] = df_final["splt_piat_ms_std"]

# Eliminiamo le colonne temporanee
colonne_da_eliminare = [
    "splt_ps_log",
    "splt_piat_ms_log",
    "splt_ps_std",
    "splt_piat_ms_std",
    "splt_direction_encoded"
]

df_final = df_final.drop(
    columns=colonne_da_eliminare,
    errors="ignore"
)

print("Dataset finale preparato.")
print("Flussi:", len(df_final))
print("Colonne:", len(df_final.columns))

Dataset finale preparato.
Flussi: 562549
Colonne: 66


In [ ]:
print("===== CONTROLLO FINALE E-6 =====")

print("\nNumero flussi:")
print(len(df_final))

print("\nDistribuzione split:")
print(df_final["split"].value_counts())

print("\nClassi label_20:")
print(df_final["label_20"].nunique())

print("\nClassi label_2:")
print(df_final["label_2"].nunique())

colonne_obbligatorie = [
    "splt_direction",
    "splt_ps",
    "splt_piat_ms",
    "splt_mask",
    "label_20",
    "label_2",
    "split"
]

print("\nControllo colonne obbligatorie:")

for col in colonne_obbligatorie:
    print(col, "→", col in df_final.columns)

===== CONTROLLO FINALE E-6 =====

Numero flussi:
562549

Distribuzione split:
split
train    393783
val       84383
test      56255
probe     28128
Name: count, dtype: int64

Classi label_20:
20

Classi label_2:
2

Controllo colonne obbligatorie:
splt_direction → True
splt_ps → True
splt_piat_ms → True
splt_mask → True
label_20 → True
label_2 → True
split → True


In [ ]:
import os

OUTPUT_PATH = "/content/drive/MyDrive/Neural Collapse/USTC-TFC2016/processed/ustc_tfc2016_preprocessed.parquet"

print("Salvataggio dataset preprocessato...")
print(OUTPUT_PATH)

df_final.to_parquet(
    OUTPUT_PATH,
    index=False,
    engine="pyarrow"
)

print("\nDataset E-6 salvato correttamente!")
print("File:", OUTPUT_PATH)
print("Flussi:", len(df_final))
print("Colonne:", len(df_final.columns))

Salvataggio dataset preprocessato...
/content/drive/MyDrive/Neural Collapse/USTC-TFC2016/processed/ustc_tfc2016_preprocessed.parquet


NameError: name 'df_final' is not defined